# Phase 2: Random Forest Baseline

Reproduces the baseline RF classifier from the **Elliptic++** paper (KDD '23).

| Step | Transactions | Actors |
|---|---|---|
| **Preprocessing** | Drop NaN, MinMaxScaler on features | Drop Time step + dupes, MinMaxScaler |
| **Unknown class** | Removed (class 3) | Removed (class 3) |
| **Labels** | licit(2) -> 0, illicit(1) -> 1 | licit(2) -> 0, illicit(1) -> 1 |
| **Split** | Timestep: train < 35, test >= 35 | 70/30 (shuffle=False, rs=15) |
| **Model** | RandomForest(n_estimators=50) | RandomForest(n_estimators=50) |
| **Features** | 183 features | 56 features |

Reference: [Elliptic++ GitHub](https://github.com/git-disl/EllipticPlusPlus)

In [1]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import logging

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import MinMaxScaler

from src.models.baseline import (
    load_transaction_features, load_actor_features,
    preprocess_transactions, preprocess_actors,
    split_transactions, split_actors,
    train_rf_transactions, train_rf_actors, ModelResults,
)
from src.explain.importance import (
    get_feature_importance, get_permutation_importance,
    get_shap_values, plot_top_features, plot_shap_summary,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
RESULTS_DIR = '../docs'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Environment loaded')

Environment loaded


## 1. Data Loading & Exploration

In [2]:
tx_raw = load_transaction_features()
actor_raw = load_actor_features()

print('=== Raw Data ===')
print(f'Transactions: {tx_raw.shape[0]:,} rows x {tx_raw.shape[1]} columns')
print(f'  Class distribution: {tx_raw["class"].value_counts().sort_index().to_dict()}')
print(f'  NaN rows: {tx_raw.isnull().any(axis=1).sum():,}')
print()
print(f'Actors: {actor_raw.shape[0]:,} rows x {actor_raw.shape[1]} columns')
print(f'  Class distribution: {actor_raw["class"].value_counts().sort_index().to_dict()}')
print(f'  Duplicate rows: {actor_raw.duplicated().sum():,}')

=== Raw Data ===
Transactions: 203,769 rows x 185 columns
  Class distribution: {1: 4545, 2: 42019, 3: 157205}
  NaN rows: 965

Actors: 1,268,260 rows x 58 columns
  Class distribution: {1: 28601, 2: 338871, 3: 900788}


  Duplicate rows: 347,569


## 2. Preprocessing

In [3]:
tx = preprocess_transactions(tx_raw)
print('=== Transactions (after preprocessing) ===')
print(f'Shape: {tx.shape}')
print(f'Class distribution: {tx["class"].value_counts().to_dict()}')
print(f'Illicit ratio: {tx["class"].mean():.3%}')

act = preprocess_actors(actor_raw)
print()
print('=== Actors (after preprocessing) ===')
print(f'Shape: {act.shape}')
print(f'Class distribution: {act["class"].value_counts().to_dict()}')
print(f'Illicit ratio: {act["class"].mean():.3%}')

=== Transactions (after preprocessing) ===
Shape: (46045, 185)
Class distribution: {0: 41500, 1: 4545}
Illicit ratio: 9.871%



=== Actors (after preprocessing) ===
Shape: (265354, 57)
Class distribution: {0: 251088, 1: 14266}
Illicit ratio: 5.376%


## 3. Train / Test Splits

In [4]:
X_train_tx, X_test_tx, y_train_tx, y_test_tx = split_transactions(tx)
print('=== Transaction Splits ===')
print(f'Train: {len(X_train_tx):,} ({X_train_tx.shape[1]} features)')
print(f'Test:  {len(X_test_tx):,}')
print(f'Train illicit ratio: {y_train_tx.mean():.3%}')
print(f'Test illicit ratio:  {y_test_tx.mean():.3%}')

X_train_act, X_test_act, y_train_act, y_test_act = split_actors(act)
print()
print('=== Actor Splits ===')
print(f'Train: {len(X_train_act):,} ({X_train_act.shape[1]} features)')
print(f'Test:  {len(X_test_act):,}')
print(f'Train illicit ratio: {y_train_act.mean():.3%}')
print(f'Test illicit ratio:  {y_test_act.mean():.3%}')

=== Transaction Splits ===
Train: 29,699 (182 features)
Test:  16,346
Train illicit ratio: 11.657%
Test illicit ratio:  6.625%

=== Actor Splits ===
Train: 185,747 (55 features)
Test:  79,607
Train illicit ratio: 6.125%
Test illicit ratio:  3.629%


## 4. Random Forest Training - Transactions

In [5]:
tx_results = train_rf_transactions(n_estimators=50, random_state=42)
print(tx_results.report)

INFO: Loading transaction data...


INFO: Splitting by timestep (train < 35, test >= 35)...


INFO: Training RF: 29699 samples, 182 features | Test: 16346 samples


INFO: RF Transactions — Precision: 0.968, Recall: 0.720, F1: 0.826, Micro-F1: 0.980


              precision    recall  f1-score   support

   licit (0)       0.98      1.00      0.99     15263
 illicit (1)       0.97      0.72      0.83      1083

    accuracy                           0.98     16346
   macro avg       0.97      0.86      0.91     16346
weighted avg       0.98      0.98      0.98     16346



In [6]:
cm_tx = confusion_matrix(tx_results.y_test, tx_results.y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_tx, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Licit', 'Illicit'],
    yticklabels=['Licit', 'Illicit'],
    ax=ax, cbar_kws={'label': 'Count'},
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('RF - Transactions Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'confusion_matrix_transactions.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

Saved: ../docs/confusion_matrix_transactions.png


## 5. Random Forest Training - Actors

In [7]:
actor_results = train_rf_actors(n_estimators=50, random_state=42, model_random_state=42)
print(actor_results.report)

INFO: Loading actor data...


INFO: Splitting actors: 70/30 (shuffle=False, rs=42)...


INFO: Training RF: 185747 samples, 55 features | Test: 79607 samples


INFO: RF Actors — Precision: 0.919, Recall: 0.788, F1: 0.848, Micro-F1: 0.990


              precision    recall  f1-score   support

   licit (0)       0.99      1.00      0.99     76718
 illicit (1)       0.92      0.79      0.85      2889

    accuracy                           0.99     79607
   macro avg       0.96      0.89      0.92     79607
weighted avg       0.99      0.99      0.99     79607



In [8]:
cm_act = confusion_matrix(actor_results.y_test, actor_results.y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_act, annot=True, fmt='d', cmap='Oranges',
    xticklabels=['Licit', 'Illicit'],
    yticklabels=['Licit', 'Illicit'],
    ax=ax, cbar_kws={'label': 'Count'},
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('RF - Actors Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'confusion_matrix_actors.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

Saved: ../docs/confusion_matrix_actors.png


## 6. Feature Importance - Transactions

In [9]:
tx_fi = get_feature_importance(tx_results.model, tx_results.feature_names)
print('Top 15 Transaction Features (RF Importance):')
print(tx_fi.head(15).to_string(index=False))

plot_top_features(
    tx_fi, top_n=20,
    title='Top 20 Transaction Feature Importances (RF)',
    output_path=os.path.join(RESULTS_DIR, 'feature_importance_transactions.png'),
)

Top 15 Transaction Features (RF Importance):
             feature  importance
    Local_feature_53    0.061159
                size    0.061023
    Local_feature_55    0.052584
num_output_addresses    0.043301
    Local_feature_47    0.038359
    Local_feature_49    0.034206
    Local_feature_90    0.029824
    Local_feature_41    0.027799
     Local_feature_5    0.027544
    Local_feature_43    0.026206
    Local_feature_14    0.022210
Aggregate_feature_45    0.021885
Aggregate_feature_68    0.020972
    Local_feature_40    0.018289
Aggregate_feature_70    0.017340


INFO: Saved feature importance plot → ../docs/feature_importance_transactions.png


'../docs/feature_importance_transactions.png'

In [10]:
tx_perm = get_permutation_importance(
    tx_results.model,
    X_test_tx.values,
    y_test_tx.values,
    tx_results.feature_names,
    n_repeats=10,
)
print('Top 15 Transaction Features (Permutation Importance):')
print(tx_perm.head(15).to_string(index=False))

INFO: Computing permutation importance (n_repeats=10)...


/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/path/to/bittrace/venv/lib/python3.11/site-packages/sklearn/utils/paralle

Top 15 Transaction Features (Permutation Importance):
             feature  importance_mean  importance_std
     Local_feature_3         0.000722        0.000206
Aggregate_feature_66         0.000428        0.000166
    Local_feature_81         0.000343        0.000132
    Local_feature_80         0.000330        0.000155
Aggregate_feature_49         0.000324        0.000099
    Local_feature_93         0.000300        0.000100
    Local_feature_53         0.000300        0.000165
Aggregate_feature_71         0.000294        0.000098
    Local_feature_10         0.000281        0.000126
Aggregate_feature_45         0.000281        0.000117
                size         0.000269        0.000117
    Local_feature_49         0.000269        0.000211
Aggregate_feature_68         0.000257        0.000086
Aggregate_feature_57         0.000245        0.000119
    Local_feature_66         0.000245        0.000091


## 7. Feature Importance - Actors

In [11]:
act_fi = get_feature_importance(actor_results.model, actor_results.feature_names)
print('Top 15 Actor Features (RF Importance):')
print(act_fi.head(15).to_string(index=False))

plot_top_features(
    act_fi, top_n=20,
    title='Top 20 Actor Feature Importances (RF)',
    output_path=os.path.join(RESULTS_DIR, 'feature_importance_actors.png'),
)

Top 15 Actor Features (RF Importance):
                   feature  importance
transacted_w_address_total    0.100574
                  fees_max    0.061385
   first_block_appeared_in    0.056514
    last_block_appeared_in    0.054837
          first_sent_block    0.054414
               fees_median    0.052206
                 fees_mean    0.049914
                  fees_min    0.047307
      first_received_block    0.046497
                fees_total    0.042807
         fees_as_share_max    0.034509
      fees_as_share_median    0.031802
        fees_as_share_mean    0.026528
      blocks_btwn_txs_mean    0.022967
         fees_as_share_min    0.022040


INFO: Saved feature importance plot → ../docs/feature_importance_actors.png


'../docs/feature_importance_actors.png'

In [12]:
act_perm = get_permutation_importance(
    actor_results.model,
    X_test_act.values,
    y_test_act.values,
    actor_results.feature_names,
    n_repeats=10,
)
print('Top 15 Actor Features (Permutation Importance):')
print(act_perm.head(15).to_string(index=False))

INFO: Computing permutation importance (n_repeats=10)...


Top 15 Actor Features (Permutation Importance):
                   feature  importance_mean  importance_std
transacted_w_address_total         0.006438        0.000251
          first_sent_block         0.004807        0.000113
      first_received_block         0.003811        0.000130
                  fees_max         0.002867        0.000100
                  fees_min         0.002130        0.000125
         fees_as_share_max         0.001994        0.000159
   first_block_appeared_in         0.001804        0.000126
    last_block_appeared_in         0.001711        0.000100
               fees_median         0.000756        0.000116
                fees_total         0.000703        0.000105
                 fees_mean         0.000628        0.000069
      fees_as_share_median         0.000515        0.000146
        fees_as_share_mean         0.000345        0.000093
       fees_as_share_total         0.000281        0.000081
       blocks_btwn_txs_max         0.000273        0

## 8. SHAP Analysis (optional)

In [13]:
shap_explainer, shap_values, X_shap = get_shap_values(
    tx_results.model,
    X_test_tx.values,
    tx_results.feature_names,
    max_samples=500,
)

if shap_explainer is not None:
    plot_shap_summary(
        shap_values,
        X_shap,
        tx_results.feature_names,
        output_path=os.path.join(RESULTS_DIR, 'shap_summary_transactions.png'),
        max_display=20,
    )
    print('SHAP summary saved')
else:
    print('SHAP not available')

INFO: Computing SHAP values (max_samples=500)...


INFO: Saved SHAP summary plot → ../docs/shap_summary_transactions.png


SHAP summary saved


## 9. ROC Curves

In [14]:
tx_probs = tx_results.model.predict_proba(X_test_tx.values)[:, 1]
tx_fpr, tx_tpr, _ = roc_curve(tx_results.y_test, tx_probs)
tx_auc = roc_auc_score(tx_results.y_test, tx_probs)

act_probs = actor_results.model.predict_proba(X_test_act.values)[:, 1]
act_fpr, act_tpr, _ = roc_curve(actor_results.y_test, act_probs)
act_auc = roc_auc_score(actor_results.y_test, act_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(tx_fpr, tx_tpr, color='#3498db', lw=2,
             label=f'RF Transactions (AUC={tx_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC - Transactions')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(act_fpr, act_tpr, color='#e67e22', lw=2,
             label=f'RF Actors (AUC={act_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC - Actors')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

fig.suptitle('ROC Curves - Random Forest Baseline', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'roc_curves.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

Saved: ../docs/roc_curves.png


## 10. Paper Comparison

In [15]:
results = {
    'Transactions (our RF)': {
        'Precision': round(tx_results.precision, 3),
        'Recall':    round(tx_results.recall, 3),
        'F1':        round(tx_results.f1, 3),
        'AUC-ROC':   round(tx_auc, 3),
    },
    'Transactions (paper RF)': {
        'Precision': 0.986,
        'Recall':    0.727,
        'F1':        0.833,
        'AUC-ROC':   0.986,
    },
    'Actors (our RF)': {
        'Precision': round(actor_results.precision, 3),
        'Recall':    round(actor_results.recall, 3),
        'F1':        round(actor_results.f1, 3),
        'AUC-ROC':   round(act_auc, 3),
    },
    'Actors (paper RF)': {
        'Precision': 0.921,
        'Recall':    0.802,
        'F1':        0.857,
        'AUC-ROC':   0.980,
    },
}
comparison = pd.DataFrame(results).T
print('Results Comparison')
print(comparison.to_string())

Results Comparison
                         Precision  Recall     F1  AUC-ROC
Transactions (our RF)        0.968   0.720  0.826    0.919
Transactions (paper RF)      0.986   0.727  0.833    0.986
Actors (our RF)              0.919   0.788  0.848    0.986
Actors (paper RF)            0.921   0.802  0.857    0.980


## 11. Summary

In [16]:
print('=' * 60)
print('PHASE 2 COMPLETE - Random Forest Baseline')
print('=' * 60)
print()
print('Transactions RF:')
print(f'   Precision: {tx_results.precision:.3f} | Recall: {tx_results.recall:.3f} | F1: {tx_results.f1:.3f}')
print(f'   AUC-ROC: {tx_auc:.3f}')
print()
print('Actors RF:')
print(f'   Precision: {actor_results.precision:.3f} | Recall: {actor_results.recall:.3f} | F1: {actor_results.f1:.3f}')
print(f'   AUC-ROC: {act_auc:.3f}')
print()
print('Output artifacts in docs/')
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith('.png'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1024
        print(f'   -> {f} ({size:.0f} KB)')
print()
print('Next: Phase 3 - GNN models (GraphSAGE, GAT)')
print('   Requires approval before proceeding.')

PHASE 2 COMPLETE - Random Forest Baseline

Transactions RF:
   Precision: 0.968 | Recall: 0.720 | F1: 0.826
   AUC-ROC: 0.919

Actors RF:
   Precision: 0.919 | Recall: 0.788 | F1: 0.848
   AUC-ROC: 0.986

Output artifacts in docs/
   -> confusion_matrix_actors.png (31 KB)
   -> confusion_matrix_transactions.png (31 KB)
   -> feature_importance_actors.png (86 KB)
   -> feature_importance_transactions.png (92 KB)
   -> roc_curves.png (115 KB)
   -> shap_summary_transactions.png (94 KB)

Next: Phase 3 - GNN models (GraphSAGE, GAT)
   Requires approval before proceeding.
